In [1]:
import pandas as pd
import duckdb

df = pd.read_parquet("../data/processed/transactions.parquet")
print(df.shape)

(1067371, 8)


In [2]:
sebelum = len(df)
df = df.drop_duplicates()
print(f"Dibuang {sebelum - len(df):,} baris dobel, sisa {len(df):,}")

Dibuang 34,335 baris dobel, sisa 1,033,036


In [3]:
is_batal = df["invoice"].str.startswith("C")
batal = df[is_batal].copy()
jual = df[~is_batal].copy()
print(f"Pembatalan: {len(batal):,} baris | Penjualan: {len(jual):,} baris")

Pembatalan: 19,104 baris | Penjualan: 1,013,932 baris


In [4]:
print("Quantity <= 0 di penjualan:", (jual["quantity"] <= 0).sum())
jual = jual[jual["quantity"] > 0]
print(f"Sisa: {len(jual):,}")

Quantity <= 0 di penjualan: 3393
Sisa: 1,010,539


In [5]:
print("Harga <= 0:", (jual["price"] <= 0).sum())
jual = jual[jual["price"] > 0]
print(f"Sisa: {len(jual):,}")

Harga <= 0: 2626
Sisa: 1,007,913


In [6]:
pola_produk = jual["stockcode"].str.match(r"^\d{5}")
print("Kode non-produk di penjualan:", (~pola_produk).sum())
jual = jual[pola_produk]

batal = batal[batal["stockcode"].str.match(r"^\d{5}")]

sisa = jual["description"].str.contains("bad debt|bank charges", case=False, na=False).sum()
print("Sisa baris piutang / biaya bank:", sisa)
print(f"Penjualan: {len(jual):,} | Pembatalan: {len(batal):,}")

Kode non-produk di penjualan: 4699
Sisa baris piutang / biaya bank: 0
Penjualan: 1,003,214 | Pembatalan: 17,914


In [7]:
jual["line_total"] = jual["quantity"] * jual["price"]
batal["line_total"] = batal["quantity"] * batal["price"]
print(f"Total penjualan: £{jual['line_total'].sum():,.0f}")
print(f"Total dibatalkan: £{-batal['line_total'].sum():,.0f}")

Total penjualan: £19,642,692
Total dibatalkan: £716,426


In [8]:
jual[["quantity", "price", "line_total"]].describe()

,quantity,price,line_total
count,1.003214e+06,1.003214e+06,1.003214e+06
mean,1.115193e+01,3.339875e+00,1.957976e+01
std,1.287471e+02,4.754713e+00,1.999959e+02
min,1.000000e+00,3.000000e-02,6.000000e-02
25%,1.000000e+00,1.250000e+00,4.130000e+00
50%,4.000000e+00,2.100000e+00,1.008000e+01
75%,1.200000e+01,4.130000e+00,1.770000e+01
max,8.099500e+04,1.157150e+03,1.684696e+05


In [9]:
jual.nlargest(5, "quantity")[["invoice", "stockcode", "description", "quantity", "invoice_date", "customer_id"]]

,invoice,stockcode,description,quantity,invoice_date,customer_id
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,16446
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,12346
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,13902
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,13902
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,13902


In [10]:
top = jual.nlargest(5, "quantity")
batal[batal["customer_id"].isin(top["customer_id"])].nsmallest(5, "quantity")[
    ["invoice", "stockcode", "description", "quantity", "invoice_date", "customer_id"]
]

,invoice,stockcode,description,quantity,invoice_date,customer_id
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,16446
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,12346
93676,C498151,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,-2504,2010-02-17 10:37:00,13902
19479,C490993,37495,FAIRY CAKE BIRTHDAY CANDLE SET,-3,2009-12-08 17:13:00,13902
19480,C490993,21463,MIRRORED DISCO BALL,-1,2009-12-08 17:13:00,13902


In [11]:
jual = jual.reset_index(drop=True)
jual["line_id"] = jual.index

kunci_batal = (
    batal.dropna(subset=["customer_id"])
         .assign(quantity=lambda d: -d["quantity"])
         [["customer_id", "stockcode", "quantity", "invoice_date"]]
         .rename(columns={"invoice_date": "tanggal_batal"})
)

pasangan = jual.merge(kunci_batal, on=["customer_id", "stockcode", "quantity"], how="inner")
pasangan = pasangan[pasangan["tanggal_batal"] >= pasangan["invoice_date"]]

print("Baris hasil merge     :", len(pasangan))
print("Baris penjualan unik  :", pasangan["line_id"].nunique())

Baris hasil merge     : 9780
Baris penjualan unik  : 9228


In [12]:
dibatalkan = pasangan["line_id"].unique()
nilai = jual.loc[jual["line_id"].isin(dibatalkan), "line_total"].sum()
print(f"{len(dibatalkan):,} baris penjualan dibatalkan penuh, senilai £{nilai:,.0f}")
print(f"= {nilai / jual['line_total'].sum():.1%} dari total penjualan")

jual = jual[~jual["line_id"].isin(dibatalkan)].drop(columns="line_id")
print(f"Sisa penjualan: {len(jual):,} baris")

9,228 baris penjualan dibatalkan penuh, senilai £731,340
= 3.7% dari total penjualan
Sisa penjualan: 993,986 baris


In [13]:
tanpa_id = jual["customer_id"].isna()
porsi_baris = tanpa_id.mean()
porsi_nilai = jual.loc[tanpa_id, "line_total"].sum() / jual["line_total"].sum()
print(f"Baris tanpa ID: {porsi_baris:.1%} dari baris, {porsi_nilai:.1%} dari nilai penjualan")

Baris tanpa ID: 22.8% dari baris, 13.6% dari nilai penjualan


In [14]:
jual_pelanggan = jual[jual["customer_id"].notna()].copy()
print(f"Semua penjualan     : {len(jual):,} baris")
print(f"Penjualan ber-ID    : {len(jual_pelanggan):,} baris")
print(f"Jumlah pelanggan    : {jual_pelanggan['customer_id'].nunique():,}")

Semua penjualan     : 993,986 baris
Penjualan ber-ID    : 767,349 baris
Jumlah pelanggan    : 5,838


In [15]:
belanja_per_pelanggan = jual_pelanggan.groupby("customer_id")["invoice"].nunique()
belanja_per_pelanggan.describe()

count    5838.000000
mean        6.212230
std        12.593771
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       367.000000
Name: invoice, dtype: float64

In [16]:
invoices = duckdb.sql("""
    WITH per_invoice AS (
        SELECT
            invoice,
            customer_id,
            MIN(invoice_date)          AS invoice_date,
            MIN(country)               AS country,
            COUNT(*)                   AS n_lines,
            SUM(quantity)              AS total_qty,
            SUM(line_total)            AS revenue
        FROM jual_pelanggan
        GROUP BY invoice, customer_id
    )
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY invoice_date, invoice) AS invoice_seq,
        LAG(invoice_date) OVER (PARTITION BY customer_id ORDER BY invoice_date, invoice) AS prev_invoice_date
    FROM per_invoice
""").df()

invoices["days_since_prev"] = (invoices["invoice_date"] - invoices["prev_invoice_date"]).dt.days
print(invoices.shape)
invoices.head()

(36267, 10)


,invoice,customer_id,invoice_date,country,n_lines,total_qty,revenue,invoice_seq,prev_invoice_date,days_since_prev
0,506394,12349,2010-04-29 13:20:00,Italy,46,473.0,1068.52,1,NaT,NaN
1,529344,12349,2010-10-28 08:23:00,Italy,54,518.0,1152.62,2,2010-04-29 13:20:00,181.0
2,577609,12349,2011-11-21 09:51:00,Italy,72,630.0,1457.55,3,2010-10-28 08:23:00,389.0
3,517512,12408,2010-07-29 13:27:00,Belgium,9,99.0,179.10,1,NaT,NaN
4,522592,12408,2010-09-15 13:16:00,Belgium,15,238.0,285.06,2,2010-07-29 13:27:00,47.0


In [17]:
contoh = invoices["customer_id"].value_counts().index[10]
invoices[invoices["customer_id"] == contoh][
    ["invoice", "invoice_date", "revenue", "invoice_seq", "prev_invoice_date", "days_since_prev"]
].head(8)

,invoice,invoice_date,revenue,invoice_seq,prev_invoice_date,days_since_prev
14207,490320,2009-12-04 15:26:00,867.89,1,NaT,NaN
14208,490324,2009-12-04 15:27:00,672.00,2,2009-12-04 15:26:00,0.0
14209,490327,2009-12-04 15:29:00,990.00,3,2009-12-04 15:27:00,0.0
14210,490725,2009-12-07 16:24:00,343.44,4,2009-12-04 15:29:00,3.0
14211,490726,2009-12-07 16:28:00,561.60,5,2009-12-07 16:24:00,0.0
14212,491812,2009-12-14 13:23:00,13854.20,6,2009-12-07 16:28:00,6.0
14213,491813,2009-12-14 13:27:00,157.50,7,2009-12-14 13:23:00,0.0
14214,491815,2009-12-14 13:36:00,177.84,8,2009-12-14 13:27:00,0.0


In [18]:
percobaan = duckdb.sql("""
    SELECT
        i.customer_id,
        SUM(i.revenue)    AS total_revenue,
        COUNT(b.invoice)  AS n_baris_batal
    FROM invoices i
    LEFT JOIN batal b ON i.customer_id = b.customer_id
    GROUP BY i.customer_id
""").df()

print(f"Total revenue versi ini   : £{percobaan['total_revenue'].sum():,.0f}")
print(f"Total revenue seharusnya  : £{invoices['revenue'].sum():,.0f}")

Total revenue versi ini   : £398,455,285
Total revenue seharusnya  : £16,337,228


In [19]:
customers = duckdb.sql("""
    WITH snapshot AS (
        SELECT CAST(MAX(invoice_date) AS DATE) + 1 AS snapshot_date
        FROM invoices
    ),
    per_customer AS (
        SELECT
            customer_id,
            ARG_MIN(country, invoice_date)   AS country,
            MIN(invoice_date)                AS first_purchase,
            MAX(invoice_date)                AS last_purchase,
            COUNT(*)                         AS n_invoices,
            SUM(revenue)                     AS total_revenue,
            AVG(revenue)                     AS avg_invoice_value,
            AVG(date_diff('day', prev_invoice_date, invoice_date)) AS avg_days_between
        FROM invoices
        GROUP BY customer_id
    ),
    batal_per_customer AS (
        SELECT
            customer_id,
            COUNT(DISTINCT invoice) AS n_cancel_invoices
        FROM batal
        WHERE customer_id IS NOT NULL
        GROUP BY customer_id
    )
    SELECT
        p.*,
        date_diff('day', CAST(p.last_purchase AS DATE), s.snapshot_date) AS recency_days,
        COALESCE(b.n_cancel_invoices, 0)                                 AS n_cancel_invoices
    FROM per_customer p
    CROSS JOIN snapshot s
    LEFT JOIN batal_per_customer b ON p.customer_id = b.customer_id
""").df()

print(customers.shape)
print(f"Total revenue versi ini   : £{customers['total_revenue'].sum():,.0f}")
print(f"Total revenue seharusnya  : £{invoices['revenue'].sum():,.0f}")
customers.head()

(5838, 10)
Total revenue versi ini   : £16,337,228
Total revenue seharusnya  : £16,337,228


,customer_id,country,first_purchase,last_purchase,n_invoices,total_revenue,avg_invoice_value,avg_days_between,recency_days,n_cancel_invoices
0,16466,United Kingdom,2010-03-07 14:31:00,2011-11-06 11:35:00,15,3642.09,242.806000,43.500000,34,1
1,16558,United Kingdom,2009-12-03 18:11:00,2011-12-09 10:20:00,38,13182.11,346.897632,19.891892,1,9
2,16573,United Kingdom,2011-02-22 15:14:00,2011-09-16 09:13:00,2,663.61,331.805000,206.000000,85,2
3,16625,United Kingdom,2010-06-14 11:56:00,2011-03-10 12:50:00,5,1654.03,330.806000,67.250000,275,1
4,16638,United Kingdom,2010-08-18 13:33:00,2011-11-17 11:09:00,15,2380.13,158.675333,32.571429,23,1


In [20]:
n_inner = duckdb.sql("""
    SELECT COUNT(*) FROM customers c
    INNER JOIN (SELECT DISTINCT customer_id FROM batal) b USING (customer_id)
""").fetchone()[0]
print(f"Pelanggan dengan LEFT JOIN : {len(customers):,}")
print(f"Pelanggan dengan INNER JOIN: {n_inner:,}")

Pelanggan dengan LEFT JOIN : 5,838
Pelanggan dengan INNER JOIN: 2,407


In [21]:
print("1 baris = 1 nota?       ", len(invoices) == invoices["invoice"].nunique())
print("1 baris = 1 pelanggan?  ", len(customers) == customers["customer_id"].nunique())

r1 = round(jual_pelanggan["line_total"].sum(), 2)
r2 = round(invoices["revenue"].sum(), 2)
r3 = round(customers["total_revenue"].sum(), 2)
print("Revenue sama di 3 tingkat?", r1 == r2 == r3, (r1, r2, r3))

print("Pelanggan yang belanja > 1 kali:", f"{(customers['n_invoices'] > 1).mean():.1%}")

1 baris = 1 nota?        True
1 baris = 1 pelanggan?   True
Revenue sama di 3 tingkat? True (np.float64(16337227.67), np.float64(16337227.67), np.float64(16337227.67))
Pelanggan yang belanja > 1 kali: 72.4%


In [22]:
folder = "../data/processed/"
jual.to_parquet(folder + "clean_lines.parquet", index=False)
batal.to_parquet(folder + "cancel_lines.parquet", index=False)
invoices.to_parquet(folder + "invoices.parquet", index=False)
customers.to_parquet(folder + "customers.parquet", index=False)
print("Tersimpan 4 file")

Tersimpan 4 file


## Ringkasan pembersihan data

| Langkah | Baris dibuang / dipisah | Sisa penjualan | Sel |
| --- | --- | --- | --- |
| Data awal | — | 1.067.371 baris (semua) | 1 |
| Buang baris dobel | 34.335 | 1.033.036 (semua) | 2 |
| Pisahkan pembatalan | 19.104 dipisah | 1.013.932 | 3 |
| Buang quantity <= 0 (koreksi stok) | 3.393 | 1.010.539 | 4 |
| Buang harga <= 0 (termasuk piutang) | 2.626 | 1.007.913 | 5 |
| Buang kode non-produk (termasuk biaya bank) | 4.699 | 1.003.214 | 6 |
| Buang pembelian yang dibatalkan penuh | 9.228 (£731.340; 3,7% nilai) | 993.986 | 11–12 |

- Baris tanpa customer_id: 22,8% baris, 13,6% nilai penjualan. Tetap ada di data penjualan, tidak dipakai untuk analisis pelanggan.
- Penjualan ber-ID: 767.349 baris, 36.267 nota, 5.838 pelanggan, total £16.337.228.
- 72,4% pelanggan belanja lebih dari sekali.
- Validasi: total revenue sama di tingkat baris, nota, dan pelanggan.

In [23]:
folder = "../data/processed/"
nama_file = ["clean_lines", "cancel_lines", "invoices", "customers"]

for nama in nama_file:
    tabel = pd.read_parquet(folder + nama + ".parquet")
    print(f"{nama:<13} {len(tabel):>9,} baris")

clean = pd.read_parquet(folder + "clean_lines.parquet")
tanpa_id = clean["customer_id"].isna()
print(f"\nTanpa ID: {tanpa_id.mean():.1%} baris, "
      f"{clean.loc[tanpa_id, 'line_total'].sum() / clean['line_total'].sum():.1%} nilai")

clean_lines     993,986 baris
cancel_lines     17,914 baris
invoices         36,267 baris
customers         5,838 baris

Tanpa ID: 22.8% baris, 13.6% nilai
